# 01 · 72 h prediction reconciliation audit

Read-only audit of the frozen case/block schedule, original predictions, recovered block, and continuation outputs.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Read-only Gray-Box AIA72 reconciliation audit.
No GPU use, no model fitting, no file deletion, no GCS mutation.
"""

from pathlib import Path
import hashlib
import io
import json
import tarfile
import pandas as pd

OLD = Path("/home/abmoses2000/graybox_aia72_inference_v1_20261004")
OLD_PRED = OLD / "execution" / "predictions"
CONT = Path("/home/abmoses2000/graybox_aia72_inference_cont2_20261005")
CONT_PRED = CONT / "predictions"

CASES = OLD / "inputs" / "cases.csv.gz"
BLOCKS = OLD / "inputs" / "blocks.json"

REC_BLOCK = "supplementary_2026_002048_002560"
REC_ARCHIVE = OLD_PRED / "proofs" / f"{REC_BLOCK}.tar.gz"
REC_EXPECTED_SHA = "b32e976538b8637354ec4d28b8409e99c107a579b878c4ff85df51bd1f5fba20"

OUT = Path("/home/abmoses2000/graybox_aia72_reconciliation_audit_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing audit directory: {OUT}")
OUT.mkdir(parents=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def block_id_from_prediction(path):
    name = path.name
    suffix = "_predictions.csv.gz"
    if not name.endswith(suffix):
        return None
    return name[:-len(suffix)]

def read_pred(path):
    df = pd.read_csv(path, low_memory=False)
    if "forecast_case_id" not in df.columns:
        raise ValueError(f"Missing forecast_case_id: {path}")
    return df

def prediction_inventory(root, source):
    rows = []
    for p in sorted(root.glob("*_predictions.csv.gz")):
        bid = block_id_from_prediction(p)
        df = read_pred(p)
        rows.append({
            "source": source,
            "block_id": bid,
            "path": str(p),
            "rows": len(df),
            "forecast_case_id_unique": bool(df["forecast_case_id"].is_unique),
            "prediction_sha256": sha256(p),
            "columns": "|".join(df.columns.astype(str)),
        })
    return rows

print("===== FROZEN INPUTS =====")
assert CASES.is_file(), CASES
assert BLOCKS.is_file(), BLOCKS
cases = pd.read_csv(CASES, low_memory=False)
blocks = json.loads(BLOCKS.read_text())

print("cases rows:", len(cases))
print("case IDs unique:", bool(cases["forecast_case_id"].is_unique))
print("\nCases by role:")
print(cases.groupby("role", sort=False).size().to_string())

block_rows = []
for i, b in enumerate(blocks):
    block_rows.append({
        "schedule_index": i,
        "block_id": b["block_id"],
        "role": b["role"],
        "start": b["start"],
        "stop": b["stop"],
        "cases": len(b["case_ids"]),
        "case_ids_unique": len(b["case_ids"]) == len(set(b["case_ids"])),
    })
block_df = pd.DataFrame(block_rows)

print("\nblocks total:", len(block_df))
print("block IDs unique:", bool(block_df["block_id"].is_unique))
print("\nBlocks by role:")
print(block_df.groupby("role", sort=False).agg(
    blocks=("block_id", "size"),
    cases=("cases", "sum"),
).to_string())

role_frames = {
    role: g.reset_index(drop=True)
    for role, g in cases.groupby("role", sort=False)
}
slice_errors = []
for b in blocks:
    role = b["role"]
    if role not in role_frames:
        slice_errors.append((b["block_id"], "role missing from cases"))
        continue
    expected = role_frames[role].iloc[b["start"]:b["stop"]]["forecast_case_id"].astype(str).tolist()
    actual = list(map(str, b["case_ids"]))
    if expected != actual:
        slice_errors.append((b["block_id"], "slice mismatch"))
print("\nblock-to-case slice errors:", len(slice_errors))

print("\n===== ORIGINAL PROGRESS =====")
progress_path = OLD_PRED / "progress.json"
if progress_path.exists():
    progress = json.loads(progress_path.read_text())
    print(json.dumps(progress, indent=2))
else:
    progress = {}
    print("progress.json missing")

print("\n===== PREDICTION FILE INVENTORY =====")
inventory = []
inventory.extend(prediction_inventory(OLD_PRED, "original"))
inventory.extend(prediction_inventory(CONT_PRED, "cont2"))
inv = pd.DataFrame(inventory)
if inv.empty:
    raise SystemExit("No prediction files found.")

print(inv.groupby("source").agg(
    blocks=("block_id", "size"),
    rows=("rows", "sum"),
).to_string())

duplicate_block_sources = (
    inv.groupby("block_id")["source"].agg(list)
    .loc[lambda s: s.map(len) > 1]
)
print("\nduplicate block IDs across local sources:", len(duplicate_block_sources))
if len(duplicate_block_sources):
    print(duplicate_block_sources.to_string())

bad_local_case_uniqueness = inv.loc[~inv["forecast_case_id_unique"], ["source","block_id","rows"]]
print("prediction files with duplicate case IDs internally:", len(bad_local_case_uniqueness))

print("\n===== MANUALLY RECOVERED BLOCK =====")
recovered_meta = {
    "block_id": REC_BLOCK,
    "archive_path": str(REC_ARCHIVE),
    "archive_exists": REC_ARCHIVE.is_file(),
    "expected_archive_sha256": REC_EXPECTED_SHA,
}
recovered_df = None
if REC_ARCHIVE.is_file():
    recovered_meta["actual_archive_sha256"] = sha256(REC_ARCHIVE)
    recovered_meta["archive_sha_matches"] = (
        recovered_meta["actual_archive_sha256"] == REC_EXPECTED_SHA
    )
    with tarfile.open(REC_ARCHIVE, "r:gz") as tar:
        names = tar.getnames()
        recovered_meta["members"] = names
        pred_name = f"{REC_BLOCK}_predictions.csv.gz"
        src_name = f"{REC_BLOCK}_sources.json"
        if pred_name not in names or src_name not in names:
            raise ValueError(f"Recovered archive has unexpected members: {names}")
        pred_raw = tar.extractfile(pred_name).read()
        recovered_meta["predictions_sha256"] = hashlib.sha256(pred_raw).hexdigest()
        recovered_df = pd.read_csv(io.BytesIO(pred_raw), compression="gzip", low_memory=False)
        recovered_meta["rows"] = len(recovered_df)
        recovered_meta["case_ids_unique"] = bool(recovered_df["forecast_case_id"].is_unique)
print(json.dumps(recovered_meta, indent=2))

print("\n===== SCHEDULE RECONCILIATION =====")
available = set(inv["block_id"])
if recovered_df is not None:
    available.add(REC_BLOCK)

block_df["available_prediction"] = block_df["block_id"].isin(available)
summary = block_df.groupby("role", sort=False).agg(
    scheduled_blocks=("block_id","size"),
    scheduled_cases=("cases","sum"),
    available_blocks=("available_prediction","sum"),
)
print(summary.to_string())

missing = block_df.loc[~block_df["available_prediction"], ["schedule_index","block_id","role","cases"]]
print("\nmissing scheduled blocks:", len(missing))
if len(missing):
    print(missing.to_string(index=False))

unexpected = sorted(available - set(block_df["block_id"]))
print("\nunexpected prediction block IDs:", len(unexpected))
for x in unexpected:
    print("  ", x)

print("\n===== CASE-ID RECONCILIATION FOR AVAILABLE PREDICTIONS =====")
frames = []
for _, row in inv.iterrows():
    df = read_pred(Path(row["path"]))
    df = df.copy()
    df["_source"] = row["source"]
    df["_block_id"] = row["block_id"]
    frames.append(df)

if recovered_df is not None:
    rdf = recovered_df.copy()
    rdf["_source"] = "manual_recovery"
    rdf["_block_id"] = REC_BLOCK
    frames.append(rdf)

combined_available = pd.concat(frames, ignore_index=True, sort=False)
dup_case_mask = combined_available["forecast_case_id"].duplicated(keep=False)

print("available prediction rows:", len(combined_available))
print("unique available prediction case IDs:", combined_available["forecast_case_id"].nunique())
print("duplicate prediction case-ID rows:", int(dup_case_mask.sum()))

if dup_case_mask.any():
    dup = combined_available.loc[
        dup_case_mask, ["forecast_case_id","_source","_block_id"]
    ].sort_values("forecast_case_id")
    print(dup.head(30).to_string(index=False))

case_ids = set(cases["forecast_case_id"].astype(str))
pred_ids = set(combined_available["forecast_case_id"].astype(str))
print("prediction IDs outside frozen cases:", len(pred_ids - case_ids))
print("frozen case IDs without an available prediction row:", len(case_ids - pred_ids))

block_lookup = {b["block_id"]: b for b in blocks}
block_case_errors = []
for frame in frames:
    bid = frame["_block_id"].iloc[0]
    if bid not in block_lookup:
        continue
    actual = frame["forecast_case_id"].astype(str).tolist()
    expected = list(map(str, block_lookup[bid]["case_ids"]))
    if actual != expected:
        block_case_errors.append({
            "block_id": bid,
            "expected_rows": len(expected),
            "actual_rows": len(actual),
            "same_set": set(actual) == set(expected),
        })
print("available blocks with case-ID/order mismatch:", len(block_case_errors))

print("\n===== POSSIBLE REUSED / PRECOMPUTED OUTPUTS =====")
candidate_files = []
tokens = ("accepted", "selection", "reuse", "precomputed", "master", "prediction")
for root in [OLD / "inputs", OLD_PRED, OLD / "execution"]:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        low = p.name.lower()
        if any(t in low for t in tokens):
            if p.suffix.lower() in {".json", ".csv", ".gz", ".parquet"}:
                candidate_files.append(str(p))
print("candidate files:", len(candidate_files))
for p in sorted(set(candidate_files))[:100]:
    print(" ", p)

block_df.to_csv(OUT / "block_schedule_audit.csv", index=False)
inv.to_csv(OUT / "prediction_file_inventory.csv", index=False)
missing.to_csv(OUT / "missing_blocks.csv", index=False)

audit = {
    "status": "READ_ONLY_RECONCILIATION_AUDIT_COMPLETE",
    "frozen_case_rows": int(len(cases)),
    "frozen_case_ids_unique": bool(cases["forecast_case_id"].is_unique),
    "scheduled_blocks": int(len(block_df)),
    "block_slice_errors": int(len(slice_errors)),
    "local_prediction_files": int(len(inv)),
    "available_prediction_rows_including_manual_recovery": int(len(combined_available)),
    "available_unique_prediction_case_ids": int(combined_available["forecast_case_id"].nunique()),
    "duplicate_prediction_case_id_rows": int(dup_case_mask.sum()),
    "missing_scheduled_blocks": int(len(missing)),
    "unexpected_prediction_block_ids": unexpected,
    "available_block_case_mismatches": block_case_errors,
    "recovered_block": recovered_meta,
    "scientific_acceptance": False,
    "note": "Audit only; no master table has been accepted or created.",
}
(OUT / "audit.json").write_text(json.dumps(audit, indent=2) + "\n")

print("\n===== AUDIT RESULT =====")
print(json.dumps(audit, indent=2))
print("\nSaved:", OUT / "audit.json")
print("Saved:", OUT / "block_schedule_audit.csv")
print("Saved:", OUT / "prediction_file_inventory.csv")
print("Saved:", OUT / "missing_blocks.csv")
